# Configuration and Decomposition: Qualitative Comparative Analysis (QCA) and Formal Demography

Two "recipe" approaches to thinking: which **combinations** of conditions produce an outcome? What **contributions** explain why some people live more years than others when you break it down? This tutorial walks through two analytical methods that are mature in qualitative and demographic traditions but have long lacked a convenient Python tool—they appear unrelated yet share the same fundamental intuition: rather than asking "on average, how much effect does one variable have," we ask "what **combinations** of conditions lead to the outcome" and "how much of the difference between two groups comes from each component when you break it down." The first half covers Qualitative Comparative Analysis (QCA); the second covers formal demography (life tables and decomposition).

**Qualitative Comparative Analysis (fuzzy-set QCA)** originates from Charles Ragin and is fundamentally based on set theory rather than regression. Regression assumes that effects of different independent variables are additive and interchangeable; QCA instead views social outcomes as often embodying "multiple conjunctural causation"—the same outcome may be triggered by several **different combinations of conditions**, and no single condition is necessary or sufficient. It treats each case's membership in each condition as a fuzzy set on `[0,1]`, enumerates all condition configurations, calculates the consistency (degree to which a configuration sufficiently causes the outcome) for each, then uses Boolean algebra (Quine–McCluskey) to simplify redundant configurations into the most parsimonious "sum of sufficient paths." The key premise here is that the outcome must be explainable in a set-theoretic sense by combinations of conditions; how you set the consistency threshold directly determines which configurations count—set it too high and you miss cases, set it too low and noisy configurations slip in.

**Formal demography** concerns itself with "how long people live" and "why mortality rates differ between populations." A life table takes a set of **age-specific mortality rates** `mx` and computes them column by column into `qx→lx→ndx→nLx→Tx→ex`, yielding life expectancy at each age, where `e0` at birth is what we commonly call "life expectancy." It is pure deterministic column arithmetic; the only difficulty lies in handling the conventions for `ax` in the infant and open-ended age intervals. Kitagawa decomposition addresses a comparative question: population B's crude death rate is higher than population A's; **how much of this difference comes from each age group truly having higher mortality (rate effect) versus merely B being older in age structure (composition effect)**? The two components sum exactly to the total difference, which is the defining property of decomposition.

All data are synthetic with known generation mechanisms, so we can validate the methods themselves: for QCA data where the outcome is truly generated by `(A and B) or C`, can the algorithm recover it; for demographic data where population B is designed to have higher mortality at each age and an older age structure, can the decomposition cleanly separate the two sources. Throughout, we use `socialverse`, the analysis library designed for social science, which targets R's `QCA` (Adrian Duşa) and `demography` / `DemoDecomp`—these methodological chains were essentially blank in the Python ecosystem until now.

In [1]:
# 让本 worktree 的 socialverse 优先于任何已安装版本被导入,并把工作目录切到本 notebook
# 所在目录,使生成的 fig_*.png 与 notebook 同目录(教学产物,可安全删除)。
import os
import sys

_NB_DIR = os.path.dirname(os.path.abspath(__file__)) if "__file__" in globals() else os.getcwd()
_PKG_ROOT = os.path.dirname(_NB_DIR)
if _PKG_ROOT not in sys.path:
    sys.path.insert(0, _PKG_ROOT)
os.chdir(_NB_DIR)

import matplotlib
matplotlib.use("Agg")  # 无显示环境:图直接写文件
import matplotlib.pyplot as plt
import pandas as pd

import socialverse as sv
from socialverse import datasets as ds

---
# Part One: Fuzzy-Set QCA

## Loading Data

`ds.load_qca()` provides 40 cases, with four columns all representing **fuzzy-set membership** on `[0,1]`: three conditions `A / B / C` and one outcome `Y`. Here "membership" is not binary (0/1) but rather "to what degree does this belong to the set"—for example, `A=0.64` means this case possesses condition A to a substantial degree. In data generation, `Y` is computed as `max(min(A,B), C)` with some noise added, meaning the underlying true set relationship is `(A and B) or C`. QCA's task is to **ignore this generation formula** and recover it from data alone.

In [2]:
qca_df = ds.load_qca()
print("案例数 × 列数:", qca_df.shape)
qca_df.head(6)

案例数 × 列数: (40, 5)


,case,A,B,C,Y
0,c1,0.64,0.57,0.76,0.80
1,c2,0.27,0.32,0.50,0.56
2,c3,0.04,0.59,0.53,0.57
3,c4,0.02,0.34,0.79,0.83
4,c5,0.81,0.39,0.41,0.41
5,c6,0.91,0.89,0.73,0.82


## Declaring Outcome and Data, Then Running Minimization

QCA needs only two things: which column is the outcome, and where the data lives. We enter the outcome variable name into the research state's `variables.outcome` and attach the data to `sources.datasets`, so `qca` can read everything it needs. `threshold` is the consistency cut-off—for a configuration to be deemed "sufficient," its consistency must meet this threshold; here we set it to the low value `0.5` so that sufficient configurations with some noise also qualify, to see if we can recover the original relationship.

Internally, `qca` proceeds step by step: calibrate each column as a fuzzy set on `[0,1]` → enumerate `2^3=8` condition configurations, calculate consistency for each (sufficiency `Σmin(X,Y)/ΣX`) and case count → configurations meeting the consistency threshold become "sufficient paths" → use Quine–McCluskey to simplify these configurations into the most parsimonious "sum of paths" → recalculate the full solution's consistency and coverage on the fuzzy data.

In [3]:
st = sv.StudyState()
st.write("variables", "outcome", "Y")          # 哪一列是结果
st.write("sources", "datasets", ds.load_qca())  # 数据挂进来

sv.tl.qca(st, conditions=["A", "B", "C"], outcome="Y", threshold=0.5)

qca_model = st.models["qca"]
print("解表达式 solution   :", qca_model["solution"])
print("解一致性 consistency:", qca_model["solution_consistency"])
print("解覆盖率 coverage   :", qca_model["solution_coverage"])
print("解类型              :", qca_model["solution_type"])

解表达式 solution   : C + A*B
解一致性 consistency: 0.9722
解覆盖率 coverage   : 0.9765
解类型              : conservative/complex (无 remainder 简化假设)


The solution expression is `C + A*B`—exactly equivalent to `(A and B) or C`. The algorithm has end-to-end recovered this set relationship without ever seeing the generation formula. **Solution consistency of 0.97** means these two paths together are almost always "sufficient" to produce `Y`; **solution coverage of 0.98** means they cover nearly all cases where `Y` appears. This is the form of conclusion QCA aims for: not "the coefficient for A is this much," but rather "satisfying `C`, or satisfying both `A` and `B` simultaneously, is sufficient to produce the outcome."

## Reading Each Path and the Full Truth Table

The solution comprises two paths, each with raw consistency (how sufficient this path is in itself) and raw coverage (how many outcome cases it alone can explain). `C` has broader coverage, while `A*B` covers the portion of cases where `C` does not hold. The **truth table** below lists all 8 configurations, sorted by consistency from high to low: `consistency` is sufficiency, `pri` is the more stringent PRI consistency (used to prevent a configuration from appearing "sufficient" for both the outcome and its negation), and rows with `outcome=1` are the sufficient configurations selected into the solution.

In [4]:
print("=== 各充分路径(solution paths) ===")
for p in qca_model["paths"]:
    print(f"  {p['term']:>4} | raw 一致性 = {p['raw_consistency']:.3f}"
          f" | raw 覆盖率 = {p['raw_coverage']:.3f}")

=== 各充分路径(solution paths) ===
     C | raw 一致性 = 0.981 | raw 覆盖率 = 0.841
   A*B | raw 一致性 = 0.975 | raw 覆盖率 = 0.560


In [5]:
tt = pd.DataFrame(st.diagnostics["consistency_coverage"]["truth_table"])
tt[["configuration", "n", "consistency", "pri", "outcome"]]

,configuration,n,consistency,pri,outcome
0,A*B*C,10,0.9941,0.9899,1
1,A*~B*C,4,0.9927,0.9839,1
2,~A*B*C,3,0.9908,0.9748,1
3,~A*~B*C,8,0.9822,0.9654,1
4,A*B*~C,3,0.9699,0.8920,1
5,~A*B*~C,1,0.9465,0.4595,0
6,~A*~B*~C,3,0.8377,0.2316,0
7,A*~B*~C,6,0.8075,0.1667,0


The first five rows (three configurations containing `C`, plus `A*B*C` and `A*B*~C`) all have consistency above 0.97 and are coded as sufficient configurations; whereas `~A*~B*~C`, `A*~B*~C`, and similar configurations that contain neither `C` nor satisfy `A and B` have consistency dropping to around 0.8, with even lower PRI, and are excluded from the solution. Simplification yields exactly `C + A*B`—this is what Boolean minimization does: compress a set of concrete configurations into the most parsimonious sufficiency statement.

## Visualization: Configuration Consistency vs. Case Count

Each point represents one configuration: the x-axis is the number of cases falling into that configuration, the y-axis is its sufficiency consistency. Red points are configurations coded as sufficient paths; gray points are excluded. The red dashed line marks the consistency threshold `0.5`. Visually, the selected sufficient configurations sit solidly in the high-consistency region, clearly separated from the excluded configurations by a vertical gap.

In [6]:
fig, ax = plt.subplots(figsize=(7, 4.5))
colors = ["#d62728" if r == 1 else "#7f7f7f" for r in tt["outcome"]]  # 红=充分角,灰=排除
ax.scatter(tt["n"], tt["consistency"], c=colors, s=90, edgecolor="black",
           linewidth=0.6, zorder=3)
for _, r in tt.iterrows():
    ax.annotate(r["configuration"], (r["n"], r["consistency"]),
                fontsize=7, xytext=(3, 3), textcoords="offset points")
ax.axhline(0.5, color="#d62728", ls="--", lw=1, label="consistency threshold = 0.5")
ax.set_xlabel("cases n (membership > 0.5 in this configuration)")
ax.set_ylabel("sufficiency consistency = min-sum(X,Y) / sum(X)")
ax.set_title("fsQCA truth table: consistency vs. case count per configuration")
ax.legend(loc="lower right", fontsize=8)
fig.tight_layout()
fig.savefig("fig_qca_truthtable.png", dpi=120, bbox_inches="tight")
plt.close(fig)
print("saved fig_qca_truthtable.png")

saved fig_qca_truthtable.png


![fsQCA Truth Table](fig_qca_truthtable.png)

---
# Part Two: Formal Demography: Life Tables and Decomposition

## Loading Data

`ds.load_demography()` provides two populations A and B with **age-specific mortality rates** `mx` and **population exposure** `pop`, divided into 9 age groups of varying widths (where `n_years` is the number of years per group). This dataset is designed so that **population B has higher mortality at each age and an older age structure**, which lets us separately illustrate both methods: the life table answers "how long does population A live," and decomposition answers "where does the mortality rate difference between B and A come from."

In [7]:
demo_df = ds.load_demography()
demo_df

,age_group,n_years,mx_A,mx_B,pop_A,pop_B
0,0,1,0.0060,0.00780,12.0,8.0
1,1-4,4,0.0004,0.00048,45.0,30.0
2,5-14,10,0.0002,0.00022,120.0,95.0
3,15-24,10,0.0009,0.00126,130.0,110.0
4,25-44,20,0.0018,0.00270,260.0,230.0
5,45-64,20,0.0080,0.01040,220.0,240.0
6,65-74,10,0.0250,0.03000,90.0,130.0
7,75-84,10,0.0700,0.07700,55.0,90.0
8,85+,15,0.1800,0.18900,25.0,55.0


## Building a Life Table and Reading Life Expectancy

`life_table` requires just three columns: age group, age-specific mortality rate, and interval width. It uses the standard Preston–Heuveline–Guillot column algorithm to compute mortality step by step into `mx → ax → qx → lx → ndx → nLx → Tx → ex`; it follows the conventional assumptions of `a0≈0.1` for the infant interval and `a=1/m` for the open-ended interval. The rightmost column `ex` represents "for people who reached this age, how many more years do they expect to live on average," and the first row's `ex` is life expectancy at birth `e0`. We first build the table for **population A**.

In [8]:
st_lt = sv.StudyState()
st_lt.write("sources", "datasets", ds.load_demography())

sv.tl.life_table(st_lt, age="age_group", mx="mx_A", width="n_years")

lt = st_lt.models["life_table"]
print("人群 A 出生时预期寿命 e0 =", round(lt["e0"], 2), "岁")
lt["table"].round(4)

人群 A 出生时预期寿命 e0 = 75.06 岁


,age,n,mx,ax,qx,lx,ndx,nLx,Tx,ex
0,0,1.0,0.0060,0.1000,0.0060,100000.0000,596.7774,9.946290e+04,7.505652e+06,75.0565
1,1-4,4.0,0.0004,2.0000,0.0016,99403.2226,158.9180,3.972951e+05,7.406189e+06,74.5065
2,5-14,10.0,0.0002,5.0000,0.0020,99244.3046,198.2903,9.914516e+05,7.008894e+06,70.6226
3,15-24,10.0,0.0009,5.0000,0.0090,99046.0143,887.4207,9.860230e+05,6.017443e+06,60.7540
4,25-44,20.0,0.0018,10.0000,0.0354,98158.5935,3471.2273,1.928460e+06,5.031420e+06,51.2581
5,45-64,20.0,0.0080,10.0000,0.1481,94687.3662,14027.7580,1.753470e+06,3.102960e+06,32.7706
6,65-74,10.0,0.0250,5.0000,0.2222,80659.6083,17924.3574,7.169743e+05,1.349490e+06,16.7307
7,75-84,10.0,0.0700,5.0000,0.5185,62735.2509,32529.3893,4.647056e+05,6.325159e+05,10.0823
8,85+,15.0,0.1800,5.5556,1.0000,30205.8615,30205.8615,1.678103e+05,1.678103e+05,5.5556


Population A has `e0 ≈ 75.06` years. In the table, `lx` starts at 100,000 (the life-table radix) and declines with age, recording "number of people still alive"; the `ex` column descends from 75 years all the way to 5.6 years in the final open-ended interval. This table is the complete product of life-table analysis; the curves of life expectancy by age group drawn below draw from it.

Next we build the same table for **population B** (with higher overall mortality), and directly compare the `e0` between the two populations.

In [9]:
st_lt.write("sources", "datasets", ds.load_demography())
sv.tl.life_table(st_lt, age="age_group", mx="mx_B", width="n_years")

lt_B = st_lt.models["life_table"]
print("人群 B 出生时预期寿命 e0 =", round(lt_B["e0"], 2), "岁")
print("A − B 的 e0 差         =", round(lt["e0"] - lt_B["e0"], 2), "岁(A 更长寿)")

人群 B 出生时预期寿命 e0 = 72.25 岁
A − B 的 e0 差         = 2.8 岁(A 更长寿)


Population B has `e0 ≈ 72.25` years, approximately 2.8 years lower than population A—consistent with the design where "B has higher mortality at each age."

## Visualization: Life Expectancy by Age Group `ex` for Both Populations

We plot the `ex` for each population across age groups. Both curves decline monotonically (older age means shorter remaining life), while population B's curve lies entirely below population A's, with the gap widest in the middle-age and elderly ranges.

In [10]:
tb_A = lt["table"]
ex_B = list(lt_B["ex"].values())

fig, ax = plt.subplots(figsize=(7, 4.5))
xpos = range(len(tb_A))
ax.plot(xpos, tb_A["ex"], "-o", color="#1f77b4", label=f"population A (e0={lt['e0']:.1f})")
ax.plot(xpos, ex_B, "-s", color="#d62728", label=f"population B (e0={lt_B['e0']:.1f})")
ax.set_xticks(list(xpos))
ax.set_xticklabels(tb_A["age"], rotation=45, ha="right", fontsize=8)
ax.set_xlabel("age group")
ax.set_ylabel("life expectancy ex (years)")
ax.set_title("Period life table: remaining life expectancy ex (A vs. B)")
ax.legend()
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig("fig_lifetable_ex.png", dpi=120, bbox_inches="tight")
plt.close(fig)
print("saved fig_lifetable_ex.png")

saved fig_lifetable_ex.png


![Life Table ex](fig_lifetable_ex.png)

## Kitagawa Decomposition: Does the Difference Come from Mortality Rates or Age Structure?

The **crude death rate** (population-wide average mortality rate) differs between the two populations for possibly two reasons: each age group truly has higher mortality, or the population is simply older with a higher proportion of elderly. Kitagawa decomposition splits the additive difference `crude_B − crude_A` into two components (let `c` denote age composition share `pop/Σpop`): the **rate effect** `Σ(mB−mA)·(cA+cB)/2` attributes to differences in mortality rates themselves, and the **composition effect** `Σ(cB−cA)·(mA+mB)/2` attributes to differences in age structure. The two components sum exactly to the total difference, with residual approximately zero—this is the validity check for decomposition.

`decomposition` by default uses the `mx_A/mx_B` and `pop_A/pop_B` columns in the data, requiring no additional parameters.

In [11]:
st_dec = sv.StudyState()
st_dec.write("sources", "datasets", ds.load_demography())

sv.tl.decomposition(st_dec)

dec = st_dec.models["decomposition"]
print("方法              :", dec["method"])
print("crude_A (粗死亡率):", round(dec["crude_A"], 5))
print("crude_B (粗死亡率):", round(dec["crude_B"], 5))
print("总差 total_diff   :", round(dec["total_diff"], 5))
print("  率效应   rate_effect        :", round(dec["rate_effect"], 5))
print("  构成效应 composition_effect :", round(dec["composition_effect"], 5))
print("adding-up 残差(应 ≈ 0)      :", round(dec["adding_up_residual"], 12))

方法              : Kitagawa
crude_A (粗死亡率): 0.01365
crude_B (粗死亡率): 0.02488
总差 total_diff   : 0.01123
  率效应   rate_effect        : 0.00231
  构成效应 composition_effect : 0.00892
adding-up 残差(应 ≈ 0)      : 0.0


The total difference is approximately `0.0112`; breaking it down: **the composition effect `0.0089` far exceeds the rate effect `0.0023`**. That is, population B's higher crude death rate is not primarily because each age group has higher mortality, but because B's age structure is older and high-mortality elderly groups make up a larger share. The residual is 0, with the two components summing exactly back to the total difference. This illustrates the value of decomposition: the blanket statement "B's death rate is higher" is split into two explainable, attributable components.

Decomposition also includes an **Oaxaca–Blinder** regression decomposition companion, which splits differences into endowments (composition differences) and coefficients (structural differences), a common alternative formulation of the same logic in social science—especially in labor economics' study of wage gaps.

In [12]:
if "oaxaca" in dec:
    ox = dec["oaxaca"]
    print(f"Oaxaca–Blinder: endowments = {ox['endowments']:.5f}, "
          f"coefficients = {ox['coefficients']:.5f}")

Oaxaca–Blinder: endowments = 0.00579, coefficients = 0.00544


## Visualization: Total Difference = Rate Effect + Composition Effect

A bar chart illustrates the adding-up property of "rate effect + composition effect = total difference." The composition effect bar is noticeably taller, consistent with the numbers above.

In [13]:
fig, ax = plt.subplots(figsize=(6.5, 4.2))
labels = ["rate\neffect", "composition\neffect", "total\ndiff"]
vals = [dec["rate_effect"], dec["composition_effect"], dec["total_diff"]]
bars = ax.bar(labels, vals, color=["#1f77b4", "#ff7f0e", "#2ca02c"], edgecolor="black")
for b, v in zip(bars, vals):
    ax.annotate(f"{v:.4f}", (b.get_x() + b.get_width() / 2, v),
                ha="center", va="bottom" if v >= 0 else "top", fontsize=9)
ax.axhline(0, color="black", lw=0.8)
ax.set_ylabel("contribution to crude-rate difference")
ax.set_title("Kitagawa: crude_B - crude_A = rate effect + composition effect")
fig.tight_layout()
fig.savefig("fig_kitagawa.png", dpi=120, bbox_inches="tight")
plt.close(fig)
print("saved fig_kitagawa.png")

saved fig_kitagawa.png


![Kitagawa Decomposition](fig_kitagawa.png)

## Age-Specific Contribution: Where Is the Difference Concentrated?

Decomposition not only gives totals but also leaves rate and composition contributions by age group, making it easy to pinpoint which ages the difference comes from. Looking at the `composition_contribution` column: the elderly groups 65 and above contribute disproportionately (the 75–84 group alone reaches `0.0025`), which explains why the overall picture is "composition effect dominant"—the bulk of population B's excess mortality comes from having a higher proportion in the elderly age groups.

In [14]:
comp = st_dec.diagnostics["components"]
comp.round(6)

,age,mx_A,mx_B,share_A,share_B,rate_contribution,composition_contribution
0,0,0.0060,0.00780,0.012539,0.008097,0.000019,-0.000031
1,1-4,0.0004,0.00048,0.047022,0.030364,0.000003,-0.000007
2,5-14,0.0002,0.00022,0.125392,0.096154,0.000002,-0.000006
3,15-24,0.0009,0.00126,0.135841,0.111336,0.000044,-0.000026
4,25-44,0.0018,0.00270,0.271682,0.232794,0.000227,-0.000087
5,45-64,0.0080,0.01040,0.229885,0.242915,0.000567,0.000120
6,65-74,0.0250,0.03000,0.094044,0.131579,0.000564,0.001032
7,75-84,0.0700,0.07700,0.057471,0.091093,0.000520,0.002471
8,85+,0.1800,0.18900,0.026123,0.055668,0.000368,0.005451


---
## Reproducible Evidence Chain

The two sections above each used independent `StudyState` instances. Beyond just producing results, `socialverse` records a provenance ledger in each state—which function was used at each step, what output was written to which slot. For social science, "which data does this conclusion come from, and through which steps" is often as important as the conclusion itself; this ledger makes the entire analysis chain traceable and reproducible. Below we print `summary()` for each.

In [15]:
print("=== QCA 链 ===")
print(st.summary())
print("\n=== 生命表链 ===")
print(st_lt.summary())
print("\n=== 分解链 ===")
print(st_dec.summary())

=== QCA 链 ===
StudyState
  sources: datasets
  variables: outcome
  models: qca
  diagnostics: consistency_coverage
  provenance: 1 step(s)

=== 生命表链 ===
StudyState
  sources: datasets
  models: life_table
  provenance: 2 step(s)

=== 分解链 ===
StudyState
  sources: datasets
  models: decomposition
  diagnostics: components
  provenance: 1 step(s)


## Summary

This tutorial has walked through two methodological chains from social science that long lacked tools in the Python ecosystem. **fsQCA** parallels R's `QCA` / `SetMethods`: it is not regression but set-theoretic configuration analysis, answering "which combinations of conditions are sufficient to produce the outcome"—we showed it can, without seeing the generation formula, fully recover `(A and B) or C` from fuzzy membership scores. **Life tables and Kitagawa decomposition** parallel R's `demography` / `DemoDecomp` and `oaxaca`: the life table yields life expectancy at each age and `e0`, while decomposition precisely splits the crude death rate difference between populations into rate and composition effects and pinpoints specific age ranges.

Compared to scattered scripts, `socialverse` adds two things: it brings these methods—previously mature only in R—onto a unified `StudyState`, sharing one set of data and plotting interfaces; and it provides an evidence chain that runs through the entire workflow, keeping a record of inputs and outputs at each step, enabling reproducibility. The next tutorial, [16_networks_stylometry](16_networks_stylometry.ipynb), shifts toward network analysis and stylometry.